In [1]:
%%capture cap
%run desp-authentication.py

Username:  qinghuai925
Password:  ········


In [2]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

In [3]:
from pyproj import datadir
import earthkit.data
import earthkit.plots
import earthkit.regrid
from polytope.api import Client

import os
import os.path  # for basename
import sys
import glob
from pathlib import Path
import warnings
import datetime
import time  # for sleep

import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from netCDF4 import Dataset
from PIL import Image

from mpl_toolkits.basemap import Basemap

import rasterio
from rasterio.transform import from_origin
from rasterio.merge import merge

from osgeo import gdal, osr, ogr  # Python bindings for GDAL
from global_land_mask import globe
from scipy.interpolate import griddata

from tqdm.auto import tqdm
from dask.diagnostics import ProgressBar

# utilities you mentioned using
from numpy import savetxt
import calendar

/work/data/zhang/anaconda3/envs/earthkit/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [4]:
# repo root: NOCOSRIOgit
repo_root = Path.cwd().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [5]:
from common import download_icedata_c
from common import data_fetching
from common import config
from common import processing
from common.download_icedata_c import request_icedata_subarea

In [6]:
climateDTmodel='ICON'
simulationperiod='future'
mapregion='Arctic'
shipclass = 'PC5'   # unchanged
datastoragedir = '/work/data/zhang/NOCOSRIOgit/usecases/RIO/data'
os.makedirs(datastoragedir, exist_ok=True)   # <-- create parents if missing
plotsdir = str(Path(datastoragedir).parent / "plots")
os.makedirs(plotsdir, exist_ok=True)


In [7]:
def compute_rio_for_date(date_str):
    if simulationperiod=='historical':
        # ICON (resolution=high) starts "1991-03-01" ends 2019-12-31
        if climateDTmodel=='ICON' and not(year in range (1991,2020)):
            raise RuntimeWarning("Historical data for ICON is currently only available between 1991-03-01 and 2019-12-31")
        # ICON (resolution=standard) starts "1990-01-01" ends "2002-02-28"
        if climateDTmodel=='ICON' and not(year in range (1990,2003)):
            raise RuntimeWarning("Historical data for ICON is currently only available between 1990-01-01 and 2002-02-28 and in standard resolution!")
    elif simulationperiod=='future':
        # ICON ("resolution": "high") starts "2020-09-01" ends "2039-12-31"
        if climateDTmodel=='ICON' and not(year in range (2020,2040)):
            raise RuntimeWarning("Future data for ICON is currently only available between 2020-09-01 and 2039-12-31")
        # ICON ("resolution": "high") starts "2020-01-01" ends on "2039-12-31"
        if climateDTmodel=='ICON' and not(year in range (2020,2040)):
            raise RuntimeWarning("Future data for ICON is currently only available between 2020-01-01 and 2039-12-31")


    # Set activity and experiment according to user input
    if simulationperiod=='historical':
        REQactivity="CMIP6"
        REQexperiment="hist"
    elif simulationperiod=='future':
        REQactivity="ScenarioMIP"
        REQexperiment="SSP3-7.0"

    # Retrieve data (SIC, sea ice velocity u, sea ice velocity v)
    dataICE1month=request_icedata_subarea(activity=REQactivity,experiment=REQexperiment,model=climateDTmodel,
                                    date=date_str,subarea=mapregion, param="263003/263004",
                                    datadir=datastoragedir,gridtype='F512')




In [ ]:
for year in range(2030, 2040):        # 2030..2039 inclusive
    for month in range(1, 13):        # Jan..Dec
        days_in_month = calendar.monthrange(year, month)[1]
        for day in range(1, days_in_month + 1):
            date_str = f"{year}{month:02d}{day:02d}"
            try:
                print(f"Processing {date_str} ...")
                compute_rio_for_date(date_str)  # expects 'YYYYMMDD'
            except Exception as e:
                print(f"[{date_str}] skipped due to error: {e}")
                continue

Processing 20300101 ...
Looking for previously downloaded data in: /work/data/zhang/NOCOSRIOgit/usecases/RIO/data
File name to look for: icedata_ScenarioMIP_SSP3-7.0_ICON_20300101_Arctic_F512_263003-263004.grb
No datadir given or no existing data file found. I will request data from Polytope...
{'activity': 'ScenarioMIP', 'class': 'd1', 'dataset': 'climate-dt', 'date': '20300101', 'experiment': 'SSP3-7.0', 'expver': '0001', 'generation': '1', 'levtype': 'o2d', 'model': 'ICON', 'param': '263003/263004', 'realization': '1', 'resolution': 'high', 'stream': 'clte', 'time': '0000', 'type': 'fc', 'grid': 'F512', 'area': '90/-180/70/180'}


2025-09-26 16:31:39 - INFO - Key read from /home/users/zhang/.polytopeapirc
2025-09-26 16:31:39 - INFO - Sending request...
{'request': 'activity: ScenarioMIP\n'
            'area: 90/-180/70/180\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20300101'\n"
            'experiment: SSP3-7.0\n'
            "expver: '0001'\n"
            "generation: '1'\n"
            'grid: F512\n'
            'levtype: o2d\n'
            'model: ICON\n'
            'param: 263003/263004\n'
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            "time: '0000'\n"
            'type: fc\n',
 'verb': 'retrieve'}
2025-09-26 16:31:39 - INFO - Polytope user key found in session cache for user zhang
2025-09-26 16:31:41 - INFO - Request accepted. Please poll ./320fd3b4-ecc2-493e-9bb9-fa5477c67605 for status
2025-09-26 16:31:41 - INFO - Polytope user key found in session cache for user zhang
2025-09-26 16:31:41 - INFO - Checki

320fd3b4-ecc2-493e-9bb9-fa5477c67605.grib:   0%|          | 0.00/351k [00:00<?, ?B/s]

2025-09-26 16:32:12 - INFO - Key read from /home/users/zhang/.polytopeapirc
2025-09-26 16:32:12 - INFO - Sending request...
{'request': 'activity: ScenarioMIP\n'
            'area: 90/-180/70/180\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20300102'\n"
            'experiment: SSP3-7.0\n'
            "expver: '0001'\n"
            "generation: '1'\n"
            'grid: F512\n'
            'levtype: o2d\n'
            'model: ICON\n'
            'param: 263003/263004\n'
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            "time: '0000'\n"
            'type: fc\n',
 'verb': 'retrieve'}
2025-09-26 16:32:12 - INFO - Polytope user key found in session cache for user zhang


Saving data to: /work/data/zhang/NOCOSRIOgit/usecases/RIO/data/icedata_ScenarioMIP_SSP3-7.0_ICON_20300101_Arctic_F512_263003-263004.grb
Processing 20300102 ...
Looking for previously downloaded data in: /work/data/zhang/NOCOSRIOgit/usecases/RIO/data
File name to look for: icedata_ScenarioMIP_SSP3-7.0_ICON_20300102_Arctic_F512_263003-263004.grb
No datadir given or no existing data file found. I will request data from Polytope...
{'activity': 'ScenarioMIP', 'class': 'd1', 'dataset': 'climate-dt', 'date': '20300102', 'experiment': 'SSP3-7.0', 'expver': '0001', 'generation': '1', 'levtype': 'o2d', 'model': 'ICON', 'param': '263003/263004', 'realization': '1', 'resolution': 'high', 'stream': 'clte', 'time': '0000', 'type': 'fc', 'grid': 'F512', 'area': '90/-180/70/180'}


...

2025-09-26 16:33:02 - INFO - Request accepted. Please poll ./8dd588c3-0755-4464-9793-7fe4c17b705f for status
2025-09-26 16:33:02 - INFO - Polytope user key found in session cache for user zhang
2025-09-26 16:33:02 - INFO - Checking request status (8dd588c3-0755-4464-9793-7fe4c17b705f)...
2025-09-26 16:33:02 - INFO - The current status of the request is 'queued'
2025-09-26 16:33:03 - INFO - The current status of the request is 'processing'
2025-09-26 16:33:07 - INFO - The current status of the request is 'processed'


8dd588c3-0755-4464-9793-7fe4c17b705f.grib:   0%|          | 0.00/373k [00:00<?, ?B/s]

2025-09-26 16:33:07 - INFO - Key read from /home/users/zhang/.polytopeapirc
2025-09-26 16:33:07 - INFO - Sending request...
{'request': 'activity: ScenarioMIP\n'
            'area: 90/-180/70/180\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20300107'\n"
            'experiment: SSP3-7.0\n'
            "expver: '0001'\n"
            "generation: '1'\n"
            'grid: F512\n'
            'levtype: o2d\n'
            'model: ICON\n'
            'param: 263003/263004\n'
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            "time: '0000'\n"
            'type: fc\n',
 'verb': 'retrieve'}
2025-09-26 16:33:07 - INFO - Polytope user key found in session cache for user zhang


Saving data to: /work/data/zhang/NOCOSRIOgit/usecases/RIO/data/icedata_ScenarioMIP_SSP3-7.0_ICON_20300106_Arctic_F512_263003-263004.grb
Processing 20300107 ...
Looking for previously downloaded data in: /work/data/zhang/NOCOSRIOgit/usecases/RIO/data
File name to look for: icedata_ScenarioMIP_SSP3-7.0_ICON_20300107_Arctic_F512_263003-263004.grb
No datadir given or no existing data file found. I will request data from Polytope...
{'activity': 'ScenarioMIP', 'class': 'd1', 'dataset': 'climate-dt', 'date': '20300107', 'experiment': 'SSP3-7.0', 'expver': '0001', 'generation': '1', 'levtype': 'o2d', 'model': 'ICON', 'param': '263003/263004', 'realization': '1', 'resolution': 'high', 'stream': 'clte', 'time': '0000', 'type': 'fc', 'grid': 'F512', 'area': '90/-180/70/180'}


2025-09-26 16:33:08 - INFO - Request accepted. Please poll ./5a3e8b64-196b-4311-b1d5-dffe238c7eb9 for status
2025-09-26 16:33:08 - INFO - Polytope user key found in session cache for user zhang
2025-09-26 16:33:08 - INFO - Checking request status (5a3e8b64-196b-4311-b1d5-dffe238c7eb9)...
2025-09-26 16:33:09 - INFO - The current status of the request is 'queued'
2025-09-26 16:33:10 - INFO - The current status of the request is 'processing'


In [ ]:
climateDTmodel='ICON'
simulationperiod='historical'
mapregion='Arctic'
shipclass = 'PC5'   # unchanged
datastoragedir = '/work/data/zhang/NOCOSRIOgit/usecases/RIO/data'
os.makedirs(datastoragedir, exist_ok=True)   # <-- create parents if missing
plotsdir = str(Path(datastoragedir).parent / "plots")
os.makedirs(plotsdir, exist_ok=True)


In [ ]:
for year in range(1990, 2020):        # 2030..2039 inclusive
    for month in range(1, 13):        # Jan..Dec
        days_in_month = calendar.monthrange(year, month)[1]
        for day in range(1, days_in_month + 1):
            date_str = f"{year}{month:02d}{day:02d}"
            try:
                print(f"Processing {date_str} ...")
                compute_rio_for_date(date_str)  # expects 'YYYYMMDD'
            except Exception as e:
                print(f"[{date_str}] skipped due to error: {e}")
                continue
